<a href="https://colab.research.google.com/github/PranshuGhori/flightdata-eda/blob/main/Flight_Delay_EDA_2019_2023.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#  **Flight Delay and Cancellation Dataset (2019-2023) - Exploratory Data Analysis**

https://www.kaggle.com/datasets/patrickzel/flight-delay-and-cancellation-dataset-2019-2023

##Download the Dataset

In [ ]:
import kagglehub

path = kagglehub.dataset_download("patrickzel/flight-delay-and-cancellation-dataset-2019-2023")

print("Path to dataset files:", path)

In [ ]:
import kagglehub
import pandas as pd
import seaborn as sns
import os

# Download dataset
path = kagglehub.dataset_download(
    "patrickze/flight-delay-and-cancellation-dataset-2019-2023"
)

print("Dataset path:", path)
print("Files:", os.listdir(path))

# Load first CSV found
csv_files = [f for f in os.listdir(path) if f.endswith(".csv")]
df = pd.read_csv(os.path.join(path, csv_files[0]))

df.head()


In [ ]:
df = pd.read_csv("/kaggle/input/flight-delay-and-cancellation-dataset-2019-2023/flights.csv")




---



---



##Data Preparation

In [ ]:
df.info()

In [ ]:
df.describe()



---



---



##**Exploratory Analysis and Visualization**




---



**1. How many total flights are in the dataset, and how many per year/month?**

The dataset contains 3 million flight observations, each representing one scheduled flight, including information on flight times, delays, cancellations, and causes.

In [ ]:
df.info()



---



**2. Which airlines operate the most flights overall?**

The dataset is dominated by major carriers, with Southwest Airlines (576K flights) representing nearly 19% of all flights, followed by Delta and American Airlines.
Regional carriers such as SkyWest, Republic, and Envoy also contribute significantly, reflecting their role as connector or feeder airlines in the U.S. network.


In [ ]:
airlines = df['AIRLINE'].value_counts().plot(kind='barh', figsize=(10, 6))
airlines.set_title('Number of Flights by Airline')
airlines.set_xlabel('Number of Flights')
airlines.set_ylabel('Airline')
plt.show()

In [ ]:
df['AIRLINE'].value_counts().plot()



---



**3. Which airports have the highest number of departures and arrivals?**

The busiest airports in the U.S. dataset are Atlanta (ATL), Dallas/Fort Worth (DFW), and Chicago O’Hare (ORD).
These major hubs dominate both departure and arrival volumes, reflecting their central role in national air traffic.
Western hubs like LAX, PHX, and SEA also appear in the top 10, showing balanced nationwide activity.

In [ ]:
import matplotlib.pyplot as plt
departures = df['ORIGIN'].value_counts()
arrivals   = df['DEST'].value_counts()

airport_traffic = (departures + arrivals).sort_values(ascending=False)
airport_traffic = airport_traffic.fillna(0).astype(int)
ax = airport_traffic.head(10).plot(kind='barh', figsize=(10,6), color='skyblue')
plt.title('Top 10 Airports by Total Flight Traffic')
plt.xlabel('Total Flights (Departures + Arrivals)')
plt.ylabel('Airport Code')
plt.tight_layout()
plt.show()





---



**4. What percentage of flights were cancelled or diverted?**

Out of 3 million total flights, approximately 2.6% were cancelled and 0.3% were diverted.
This indicates that over 97% of flights operated successfully, showing overall strong operational reliability across U.S. airlines.

In [ ]:
(df['CANCELLED'].value_counts()/len(df))*100

In [ ]:
(df['DIVERTED'].value_counts() / len(df)) * 100




---



**5. What is the average departure delay and arrival delay across all flights?**

On average, flights depart about 10 minutes late but arrive only 4 minutes late.
This suggests that most flights make up some lost time while airborne, a common pattern in U.S. commercial aviation where airlines build small buffer times into their schedules.


In [ ]:
df[['DEP_DELAY', 'ARR_DELAY']].mean()



---



**6. What percentage of flights were on time, delayed, or early?**
The majority of flights (~59%) actually departed earlier than their scheduled time,
about 23% were on time (within 15 minutes),
and 17% experienced significant departure delays (>15 minutes).

This suggests airlines build extra buffer time into schedules, allowing many flights to push back slightly ahead of time — improving overall punctuality metrics.

In [ ]:
dep = df['DEP_DELAY'].fillna(0).astype(int)

count_early   = (dep < 0).sum()
count_on_time = ((dep >= 0) & (dep <= 15)).sum()
count_delayed = (dep > 15).sum()

total = len(dep)

print("Early:", round(count_early / total * 100, 2), "%")
print("On-time:", round(count_on_time / total * 100, 2), "%")
print("Delayed:", round(count_delayed / total * 100, 2), "%")
print("Total:", round((count_early + count_on_time + count_delayed) / total * 100, 2), "%")




---



**7. Which airlines have the highest and lowest average delays?**

JetBlue and Frontier Airlines show the highest average delays, both in departure and arrival times — over 10 minutes late on average.

Alaska Airlines, Horizon Air, and Hawaiian Airlines maintain the lowest average delays, typically departing and arriving within 5 minutes of schedule.

Endeavor Air even shows a negative mean arrival delay (-1.26 min), indicating that it often arrives early.

In [ ]:
airline_delay = (
    df.groupby('AIRLINE')[['DEP_DELAY', 'ARR_DELAY']]
      .mean()
      .sort_values('DEP_DELAY', ascending=False)
)
airline_delay




---



**8. Which airports experience the worst average delays (departure & arrival)?**

The airports with the worst on-time performance are primarily smaller regional or island airports such as PPG (Pago Pago, American Samoa) and SMX (Santa Maria, CA), both showing extremely high average delays exceeding 30–50 minutes.

These smaller airports may face:

Limited runway or ground handling capacity

Fewer daily flights, making each delay more impactful

Dependency on weather-sensitive or smaller aircraft routes

In contrast, major hubs like ATL, DFW, and DEN—despite heavy traffic—usually maintain much lower average delays because of higher resource availability and better scheduling systems.

In [ ]:
airport_delay = (
    df.groupby('ORIGIN')[['DEP_DELAY', 'ARR_DELAY']]
    .mean()
    .sort_values('DEP_DELAY', ascending=False)
)
airport_delay.head(10)



---



**9. How does the average delay vary by hour of scheduled departure (CRS_DEP_TIME)?**

Delays grow through the day. Early-morning departures (05:00–09:00) have the best punctuality and often arrive early, while evening departures (18:00–22:00) experience the highest average delays (15–17 minutes). Median delays are smaller than means (skew from a few extreme late flights). Very early hours (0–4) show high means but low volume, so a small number of severe delays can distort averages.

In [ ]:
df['SCHEDULE_HOUR'] = df['CRS_DEP_TIME'].fillna(0).astype(int) // 100
hour_delay = (
    df.groupby('SCHEDULE_HOUR')[['DEP_DELAY', 'ARR_DELAY']]
    .mean()

)
hour_delay.head(100).sort_values('SCHEDULE_HOUR', ascending=True)



---



**10. How do delays vary by day of week (Monday–Sunday)?**

Flights departing on Sundays and Thursdays experience the worst delays, while Tuesdays and Wednesdays tend to be the most punctual.
This pattern aligns with known weekly traffic cycles, where business and leisure travel peaks create operational congestion at the start and end of the week.

In [ ]:
df['DAY'] = pd.to_datetime(df['FL_DATE'])
df['DAY_NAME']= df['DAY'].dt.day_name()

day_delay = df.groupby('DAY_NAME')[['DEP_DELAY', 'ARR_DELAY']].mean()
delays_by_day = day_delay.plot(kind='barh', figsize=(10, 6));
delays_by_day.set_title('Delays by days')
delays_by_day.set_xlabel('Delays in minutes')
delays_by_day.set_ylabel('Day')




---



**11. How do delays vary by month or season (winter, summer, etc.)?**

Flight delays exhibit clear seasonality, peaking during Summer (13.8 min dep delay) and Winter (10.2 min).
These periods align with extreme weather and heavy passenger loads, while Fall remains the most reliable travel season.

In [ ]:
df['MONTH'] = df['DAY'].dt.month

months_delay = df.groupby(['MONTH'])[['DEP_DELAY', 'ARR_DELAY']].mean()

def get_season(month):
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    else:
        return 'Fall'

df['SEASON'] = df['MONTH'].apply(get_season)

season_delay = df.groupby('SEASON')[['DEP_DELAY', 'ARR_DELAY']].mean()
season_delay




---



**12. Are evening flights delayed more often than morning flights?**

Evening flights experience the longest average delays, departing about 16 minutes late and arriving over 10 minutes behind schedule.

Afternoon flights also show notable delays, likely due to cumulative congestion from earlier flights.

Morning flights are the most punctual, often departing within 5 minutes of schedule and sometimes even arriving early.

Night flights (post-midnight departures) tend to be lightly loaded and often arrive early due to lower air traffic.

The delay buildup over the day is a cascading effect — small morning delays propagate across flights, aircraft rotations, and crews.

Airports and airlines often schedule buffer time in the late evening, but it’s insufficient to offset the accumulated lag.

Passengers seeking on-time reliability should prefer early morning departures whenever possible.

In [ ]:

def time_of_day(hour):
  if hour in range(0,6):
    return 'Night'
  elif hour in range(6,12):
    return 'Morning'
  elif hour in range(12,18):
    return 'Afternoon'
  else:
    return 'Evening'

df['TIME_OF_DAY'] = df['SCHEDULE_HOUR'].apply(time_of_day)
delay_by_time_of_day = df.groupby('TIME_OF_DAY')[['DEP_DELAY', 'ARR_DELAY']].mean()
delay_by_time_of_day



---



**13. What is the average delay caused by each factor:**

**Carrier, Weather, NAS (Air Traffic Control), Security, Late aircraft**

The largest contributors to total delay are:
🔹 Late aircraft arrivals (4.53 min)
🔹 Airline-related issues (4.41 min)

Together, these two factors account for over 60% of total average delay time.

Air Traffic Control (NAS) delays are moderate, around 2.3 minutes on average.

Weather-related delays are relatively low (0.7 min), but they’re highly sporadic — a few severe storms can spike this number.

Security delays are minimal (0.03 min) — rare events like extra screening or threats.

In [ ]:
from collections.abc import AsyncGenerator
reason_of_delay = df[
    ['DELAY_DUE_CARRIER', 'DELAY_DUE_WEATHER', 'DELAY_DUE_NAS',
     'DELAY_DUE_SECURITY', 'DELAY_DUE_LATE_AIRCRAFT']
].fillna(0).mean().sort_values(ascending=False)

reason_of_delay




---



**14. Which cause contributes the most total delay minutes overall?**

The largest total delay comes from Late Aircraft (≈13.6 million minutes).

Carrier-related delays are a close second at 13.2 million minutes.

Air Traffic Control (NAS) causes around 7 million minutes, roughly half of the top two causes.

Weather adds about 2 million minutes, and Security is negligible.

In [ ]:
total_delay_by_cause = df[
    ['DELAY_DUE_CARRIER', 'DELAY_DUE_WEATHER',
     'DELAY_DUE_NAS', 'DELAY_DUE_SECURITY',
     'DELAY_DUE_LATE_AIRCRAFT']
].fillna(0).sum().sort_values(ascending=False)

total_delay_by_cause




---



**15. How do delay causes differ by airline (e.g., is weather more for some)?**

Delay causes vary notably among airlines, reflecting differences in operations, routes, and fleet structures. Overall, carrier-related and late aircraft delays are the most significant contributors across most airlines.

SkyWest Airlines shows the highest carrier-related delays (42.1 mins) and also experiences the largest weather-related delays (10.1 mins), suggesting operational sensitivity to weather and internal inefficiencies.

Mesa Airlines, PSA Airlines, and Frontier Airlines have the highest late aircraft delays (32+ mins), indicating cascading effects from previous flight delays and tight turnaround schedules.

ExpressJet and Spirit Air Lines show elevated NAS (Air Traffic Control) delays, which is common for airlines operating in heavily congested airports.

Delta Air Lines, JetBlue, and American Airlines face strong carrier delays (around 27–31 mins), pointing toward internal scheduling or ground operation inefficiencies.

Regional carriers such as Envoy Air and Endeavor Air exhibit higher weather delays, likely due to shorter routes and exposure to local climate variations.

Hawaiian Airlines consistently reports the lowest delay times across all causes, benefiting from isolated routes and stable weather conditions.

In summary, while weather and air traffic control play a role, most delays are within airline control, driven by carrier operations and late aircraft issues. Reducing turnaround times and improving fleet coordination could significantly improve on-time performance.

In [ ]:
total_delay_by_cause_airline = df[
    ['DELAY_DUE_CARRIER', 'DELAY_DUE_WEATHER',
     'DELAY_DUE_NAS', 'DELAY_DUE_SECURITY',
     'DELAY_DUE_LATE_AIRCRAFT','AIRLINE']]
delay_airline_causes = total_delay_by_cause_airline.groupby('AIRLINE')[['DELAY_DUE_CARRIER', 'DELAY_DUE_WEATHER',
     'DELAY_DUE_NAS', 'DELAY_DUE_SECURITY',
     'DELAY_DUE_LATE_AIRCRAFT']]

delay_airline_causes_plot = delay_airline_causes.mean().round(1).sort_values('DELAY_DUE_CARRIER',ascending=False)
delay_airline_causes_plot




---



**16. What are the top 10 routes (ORIGIN → DEST) with the highest average delay?**

The analysis of average delays by route shows that certain connections experience extremely high delay times, often due to isolated or infrequent flights (e.g., DEN → ABE, MIA → HSV). However, after filtering out extreme outliers, consistent delay-prone routes typically involve regional airports or busy hub pairs, reflecting both congestion and operational constraints.

This route-level insight can help airlines identify problematic city pairs for schedule adjustments or resource allocation to minimize recurring delays.


In [ ]:
df['ROUTE'] = df['ORIGIN'] + ' → ' + df['DEST']
route_delay = df.groupby('ROUTE')[['DEP_DELAY', 'ARR_DELAY']].mean().sort_values('DEP_DELAY', ascending=False)
route_delay.head(10)

In [ ]:
df_filtered = df[(df['DEP_DELAY'] < 300) & (df['ARR_DELAY'] < 300)]
route_delay = (
    df_filtered.groupby('ROUTE')[['DEP_DELAY', 'ARR_DELAY']]
    .mean()
    .sort_values('DEP_DELAY', ascending=False)
)
route_delay.head(10)




---



**17. How does average delay change with flight distance?**

Overall, flight delays increase with distance up to ~2000 miles, after which they stabilize or slightly improve.
This pattern suggests that operational complexity and airport congestion play a greater role in delays than flight duration alone.

In [ ]:
df['DISTANCE'].describe()
bins = [0, 500, 1000, 2000, 6000]
labels = ['Short-haul (<500 mi)', 'Medium-haul (500–1000 mi)', 'Long-haul (1000–2000 mi)', 'Ultra-long (>2000 mi)']

df['DISTANCE_BIN'] = pd.cut(df['DISTANCE'], bins=bins, labels=labels, include_lowest=True)
distance_delay = df.groupby('DISTANCE_BIN')[['DEP_DELAY', 'ARR_DELAY']].mean().round(2)
distance_delay

distance_delay.plot(kind='barh', figsize=(5,5))
plt.title('Average Delay by Flight Distance Category')
plt.xlabel('Flight Distance Category')
plt.ylabel('Average Delay (minutes)')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show();




---



**18. Which routes have the highest cancellation rates?**

The New York (LGA) ↔ Chicago (ORD) corridor had the highest number of cancellations, likely due to:

- Heavy air traffic between two major hubs.

- Frequent weather disruptions.

- Tight scheduling at both airports.

Other high-cancellation routes include Houston ↔ Dallas and Boston ↔ LaGuardia, also busy domestic corridors with high flight volumes.

These results represent raw counts, so routes with more total flights will naturally appear higher.

In [ ]:
cancel_by_route = (
    df.groupby('ROUTE')['CANCELLED']
      .sum()
      .sort_values(ascending=False)
)
cancel_by_route.head(10)





---



---



##Summary and Conclusion

- The dataset covers ~3M U.S. flights, with Southwest, Delta, and American Airlines operating the most.

- About 2.6% of flights were cancelled and 17% delayed; morning flights are most punctual while evening flights face the longest delays.

- Average delays: ~10 min departure and ~4 min arrival.

- Late aircraft and carrier issues are the top delay causes; weather and security contribute least.

- Summer shows the worst seasonal delays, while fall performs best.

- Routes like LGA ↔ ORD and HOU ↔ DAL see the most cancellations.

- Short-haul flights tend to be more delay-prone than longer ones.

**Conclusion:**
Delays are mainly driven by schedule congestion and late aircraft turnover, not weather.
Improving aircraft turnaround, scheduling buffers, and coordination between airports and airlines can significantly enhance on-time performance.
